In [23]:
import pypsa
from tz_pypsa.constraints import (constr_max_annual_utilisation_generator, 
                                  constr_min_annual_utilisation_generator,
                                  constr_max_annual_utilisation_links,
                                  constr_min_annual_utilisation_links,
                                  constr_max_annual_utilisation_storage_discharge,
                                  constr_min_annual_utilisation_storage_discharge,
                                  constr_max_annual_utilisation_storage_charge,
                                  constr_min_annual_utilisation_storage_charge
                                  )

import plotly.express as px
import pandas as pd     
import numpy as np
import xarray as xr
import os
os.environ['GRB_LICENSE_FILE'] = '/home/jy/opt/gurobi/gurobi.lic'

In [2]:
n = pypsa.Network()
n.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_v3/platform_network.nc")

INFO:pypsa.io:Imported network platform_network.nc has buses, carriers, generators, links, loads, storage_units


In [3]:
n.generators['carrier'] = n.generators['type']
n.links['carrier'] = n.links['type']
n.storage_units['carrier'] = n.storage_units['type']

In [4]:
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-unspecified'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-unspecified'), 'ramp_limit_down'] = 0.9

In [5]:
n.storage_units['standing_loss'] = 5e-05
n.storage_units['cyclic_state_of_charge'] = True

In [ ]:
target_regions = ([
                    "HK", #not fine
                    "TH", #not fine
                    "TK", #not fine
                     "CB", #fine
                     "HR", #fine
                    "KA",  #not fine
                    "CG", #fine
                    "SH",  #not fine
                    "KY" #not fine
                   ]
                   )
region_pattern = '|'.join(target_regions)
region_units = n.storage_units.index[n.storage_units.bus.str.contains(region_pattern)]

time_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 21
n.storage_units_t.state_of_charge_set.loc[time_mask, region_units] = np.nan

In [6]:
target_regions = ([
                    "HK", #not fine
                    "TH", #not fine
                    "TK", #not fine
                     "CB", #fine
                     "HR", #fine
                    "KA",  #not fine
                    "CG", #fine
                    "SH",  #not fine
                    "KY" #not fine
                   ]
                   )
region_pattern = '|'.join(target_regions)
region_units = n.storage_units.index[n.storage_units.bus.str.contains(region_pattern)]

time_mask = (n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 13) | (n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 5)| (n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 21)
n.storage_units_t.state_of_charge_set.loc[time_mask, region_units] = np.nan

In [ ]:
target_regions = ([
                    "HK", #not fine
                    "TH", #not fine
                    "TK", #not fine
                    "CB", #fine
                    "HR", #fine
                    "KA",  #not fine
                    "CG", #fine
                    "SH",  #not fine
                    "KY" #not fine
        
])
region_pattern = '|'.join(target_regions)
region_units = n.storage_units.index[n.storage_units.bus.str.contains(region_pattern)]

# Get masks for hour 1 and hour 2
hour_1_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 5
hour_2_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 13
hour_3_mask = n.storage_units_t.state_of_charge_set.index.get_level_values('timestep').hour == 21

# Get hour 1 values for the target regions
hour_1_values = n.storage_units_t.state_of_charge_set.loc[hour_1_mask, region_units]
hour_2_values = n.storage_units_t.state_of_charge_set.loc[hour_2_mask, region_units]
hour_3_values = n.storage_units_t.state_of_charge_set.loc[hour_3_mask, region_units]
# Multiply by 0.95 and assign to hour 2
# Note: The indices should align automatically since both masks select the same dates
n.storage_units_t.state_of_charge_set.loc[hour_1_mask, region_units] = hour_1_values.values * 0.7
n.storage_units_t.state_of_charge_set.loc[hour_2_mask, region_units] = hour_2_values.values * 0.7
n.storage_units_t.state_of_charge_set.loc[hour_3_mask, region_units] = hour_3_values.values * 0.7

In [7]:
n.storage_units_t.state_of_charge_set[n.storage_units_t.state_of_charge_set.notna().any(axis=1)]

,StorageUnit,hydro-pumped-storage-unspecified:GRIDREGION-JPN-CB,hydro-pumped-storage-unspecified:GRIDREGION-JPN-CG,hydro-pumped-storage-unspecified:GRIDREGION-JPN-HK,hydro-pumped-storage-unspecified:GRIDREGION-JPN-HR,hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA,hydro-pumped-storage-unspecified:GRIDREGION-JPN-KY,hydro-pumped-storage-unspecified:GRIDREGION-JPN-SH,hydro-pumped-storage-unspecified:GRIDREGION-JPN-TH,hydro-pumped-storage-unspecified:GRIDREGION-JPN-TK
period,timestep,,,,,,,,,


In [8]:
n.generators.loc[n.generators.carrier == 'nuclear', 'p_min_pu'] = 0.59
n.generators.loc[n.generators.carrier == 'nuclear', 'p_max_pu'] = 0.59

n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_min_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_min_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_min_pu'] = 0.585
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.585

In [ ]:
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.30
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'p_min_pu'] = -0.50

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.63
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'p_min_pu'] = -0.70

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.50
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'p_min_pu'] = -0.70

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.50
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'p_min_pu'] = -0.65

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.50
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'p_min_pu'] = -0.70

In [9]:
n.generators_t.p_min_pu = n.generators_t.p_max_pu.filter(regex='biomass|geothermal|hydro')

In [10]:
# p_max_pu
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

# p_min_pu
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_min_pu'] = 0.10
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_min_pu'] = 0.15
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_min_pu'] = 0.20
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_min_pu'] = 0.20

In [11]:
# max_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.55
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.41
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.365

# gas
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.09
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.23

In [12]:
# min_utilisation_rate
# coal
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.365

# gas
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

In [13]:
# max_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.44
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.33
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.18
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.40
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.34
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.83

In [14]:
# min_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.04
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.91

In [15]:
# max_utilisation_rate
# hydro-pumped-storage
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'discharge_min_utilisation_rate'] = 0.126
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'charge_max_utilisation_rate'] = 0.180

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'discharge_min_utilisation_rate'] = 0.136
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'charge_max_utilisation_rate'] = 0.195

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'discharge_min_utilisation_rate'] = 0.121
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'charge_max_utilisation_rate'] = 0.174

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'discharge_min_utilisation_rate'] = 0.059
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'charge_max_utilisation_rate'] = 0.085

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'discharge_min_utilisation_rate'] = 0.055
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'charge_max_utilisation_rate'] = 0.081

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'discharge_min_utilisation_rate'] = 0.064
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'charge_max_utilisation_rate'] = 0.092

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'discharge_max_utilisation_rate'] = 0.057
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'charge_max_utilisation_rate'] = 0.083

In [59]:
n.optimize.create_model()

constr_max_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Set max annual utilisation for these generators
constr_min_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Set min annual utilisation for these generators
constr_max_annual_utilisation_links(n, carriers='transmission') # Set max annual utilisation for these links
constr_min_annual_utilisation_links(n, carriers='transmission') # Set min annual utilisation for these links
constr_max_annual_utilisation_storage_discharge(n, carriers='hydro-pumped-storage-unspecified') # Set max annual utilisation for these storage units
constr_min_annual_utilisation_storage_discharge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
constr_max_annual_utilisation_storage_charge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units

Index(['hydro-pumped-storage-unspecified:GRIDREGION-JPN-SH',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-HR',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-CB',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-HK',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-CG',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-TK',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-TH',
       'hydro-pumped-storage-unspecified:GRIDREGION-JPN-KY'],
      dtype='object', name='StorageUnit')
Index(['wind-offshore-unspecified:GRIDREGION-JPN-SH',
       'wind-offshore-unspecified:GRIDREGION-JPN-HR',
       'wind-offshore-unspecified:GRIDREGION-JPN-CB',
       'wind-offshore-unspecified:GRIDREGION-JPN-HK',
       'wind-offshore-unspecified:GRIDREGION-JPN-KA',
       'wind-offshore-unspecified:GRIDREGION-JPN-CG',
       'wind-offshore-unspecified:GRIDREGION-JPN-TK',
       'wind-offshore-unspecif

['coal-unspecified:GRIDREGION-JPN-SH', 'coal-unspecified:GRIDREGION-JPN-HR', 'coal-unspecified:GRIDREGION-JPN-CB', 'coal-unspecified:GRIDREGION-JPN-HK', 'coal-unspecified:GRIDREGION-JPN-KA', 'coal-unspecified:GRIDREGION-JPN-CG', 'coal-unspecified:GRIDREGION-JPN-TK', 'coal-unspecified:GRIDREGION-JPN-TH', 'coal-unspecified:GRIDREGION-JPN-KY', 'gas-unspecified:GRIDREGION-JPN-SH', 'gas-unspecified:GRIDREGION-JPN-HR', 'gas-unspecified:GRIDREGION-JPN-KA']
['coal-unspecified:GRIDREGION-JPN-HK', 'coal-unspecified:GRIDREGION-JPN-TK', 'coal-unspecified:GRIDREGION-JPN-TH', 'coal-unspecified:GRIDREGION-JPN-KY', 'gas-unspecified:GRIDREGION-JPN-SH', 'gas-unspecified:GRIDREGION-JPN-HR', 'gas-unspecified:GRIDREGION-JPN-CB', 'gas-unspecified:GRIDREGION-JPN-HK', 'gas-unspecified:GRIDREGION-JPN-KA', 'gas-unspecified:GRIDREGION-JPN-CG', 'gas-unspecified:GRIDREGION-JPN-TK', 'gas-unspecified:GRIDREGION-JPN-TH', 'gas-unspecified:GRIDREGION-JPN-KY']
['transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH', 'transm

In [ ]:
def load_soc_bounds_as_xarray(
        min_csv: str , 
        max_csv: str ,
        unit: str,
    ) -> xr.Dataset:
    """
    Load and merge SOC bounds from flat CSVs.
    
    """
    def process_csv(filepath, var):
        df = pd.read_csv(filepath)
        df = df.loc[df.user_data.notna()]
        df['StorageUnit'] = df['technology'] + ":" + df['node']
        df['hour'] = df['hour'].astype(int)
        df = df.rename(columns={
            'user_data': var}
            )
        return df.set_index(['StorageUnit', unit])[[var]]

    # Process csv
    df_min = process_csv(min_csv, 'min_frac', unit)
    df_max = process_csv(max_csv, 'max_frac', unit)
    
    # Merge on the Index
    df_merged = pd.concat([df_min, df_max], axis=1, join='outer')

    # Convert to xarray
    ds = df_merged.to_xarray()

    print(f"Loaded SOC bounds for {ds.sizes['StorageUnit']} units and {ds.sizes['hour']} hours.")
    
    return ds

In [95]:
intraday_ds = load_soc_bounds_as_xarray(
    min_csv = "/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_min.csv",
    max_csv = "/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_intraday_profile_annual_max.csv",
)

weekly_ds = load_soc_bounds_as_xarray(
    min_csv = "/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_min.csv",
    max_csv = "/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing/templates/state_of_charge_weekly_profile_annual_max.csv",
)

Loaded SOC bounds for 9 units and 24 hours.


KeyError: 'hour'

In [ ]:
def constr_soc_intraday_profile(network, bounds_ds):
    
    # Intersection of units in the CSV and units in the Network
    common_units = list(set(bounds_ds.StorageUnit.values) & set(network.storage_units.index))
    
    if not common_units:
        print("No matching storage units found.")
        return

    # Slice the input dataset to matches
    targets = bounds_ds.sel(StorageUnit=common_units)

    # Process capacity
    p_nom = network.storage_units.loc[common_units, 'p_nom']
    max_hours = network.storage_units.loc[common_units, 'max_hours']
    capacity = xr.DataArray(
        p_nom * max_hours, 
        dims="StorageUnit", 
        coords={"StorageUnit": common_units}
    )
    
    # Create RHS (StorageUnit, Hour)
    snapshot_counts = network.snapshots.get_level_values('timestep').hour.value_counts().sort_index()
    counts_array = snapshot_counts.values
    hours_array = snapshot_counts.index.values 

    days_count = xr.DataArray(
        counts_array, 
        dims="hour", 
        coords={"hour": hours_array} 
    )
    rhs_base = capacity * days_count

    # Process LHS
    soc_vars = network.model.variables["StorageUnit-state_of_charge"].sel(StorageUnit=common_units)
    soc_hourly_sum = soc_vars.groupby("timestep.hour").sum()

    # Build min and max constraints
    if 'min_frac' in targets:
        network.model.add_constraints(
            soc_hourly_sum >= targets['min_frac'] * rhs_base,
            name="StorageUnit-intraday_soc_min",
            mask=targets['min_frac'].notnull()
        )

    if 'max_frac' in targets:
        network.model.add_constraints(
            soc_hourly_sum <= targets['max_frac'] * rhs_base,
            name="StorageUnit-intraday_soc_max",
            mask=targets['max_frac'].notnull()
        )
        
    print(f"Constraints added for {len(common_units)} units.")

In [94]:
def constr_soc_weekly_profile(
    network: pypsa.Network, 
    bounds_ds: xr.Dataset, 
    day_shift: int = 0
):
    
    # Align StorageUnits
    common_units = list(set(bounds_ds.StorageUnit.values) & set(network.storage_units.index))
    if not common_units:
        print("No matching storage units found.")
        return

    targets = bounds_ds.sel(StorageUnit=common_units)

    # Process Capacity (StorageUnit dim)
    p_nom = network.storage_units.loc[common_units, 'p_nom']
    max_hours = network.storage_units.loc[common_units, 'max_hours']
    
    capacity = xr.DataArray(
        (p_nom * max_hours).values, 
        dims="StorageUnit", 
        coords={"StorageUnit": common_units}
    )
    
    # Process LHS Variables (The Grouping Logic)
    soc_vars = network.model.variables["StorageUnit-state_of_charge"].sel(StorageUnit=common_units)
    
    # Extract MultiIndex to ensure safe alignment
    multi_index = soc_vars.indexes['snapshot']
    timestamps = multi_index.get_level_values(1) # Assumes level 1 is datetime
    
    # Calculate day integers (0=Mon, 6=Sun) with optional shift
    # If day_shift=0, this is standard dayofweek
    day_ints = (timestamps.dayofweek + day_shift) % 7
    
    grouper_name = "dayofweek" if day_shift == 0 else "shifted_dayofweek"

    # Create safe DataArray for grouping
    grouper = xr.DataArray(
        day_ints,
        dims="snapshot",              # Matches PyPSA variable dim
        coords={"snapshot": multi_index}, # Matches PyPSA variable coords
        name=grouper_name
    )
    
    # LHS: Sum SOC per day-bucket
    soc_sum = soc_vars.groupby(grouper).sum()

    # 4. Process RHS (Days Count)
    # Count occurrences of 0..6 in the shifted array
    # We must use the SAME array we used for grouping to ensure counts match
    unique_days, counts = pd.factorize(day_ints, sort=True)
    counts_map = pd.Series(index=unique_days, data=np.bincount(counts[counts >= 0]))
    
    days_count = xr.DataArray(
        counts_map.values,
        dims=grouper_name, 
        coords={grouper_name: counts_map.index.values} 
    )

    # 5. Final RHS Assembly
    # Ensure targets use the same dimension name for broadcasting
    if grouper_name != "dayofweek" and "dayofweek" in targets.dims:
         targets = targets.rename({"dayofweek": grouper_name})

    rhs_base = capacity * days_count

    # 6. Build Constraints
    if 'min_frac' in targets:
        network.model.add_constraints(
            soc_sum >= targets['min_frac'] * rhs_base,
            name=f"StorageUnit-weekly_soc_min",
            mask=targets['min_frac'].notnull()
        )

    if 'max_frac' in targets:
        network.model.add_constraints(
            soc_sum <= targets['max_frac'] * rhs_base,
            name=f"StorageUnit-weekly_soc_max",
            mask=targets['max_frac'].notnull()
        )
        
    print(f"Added weekly constraints for {len(common_units)} units (Shift={day_shift}).")

In [60]:
constr_soc_intraday_profile(n, bounds_ds=ds)

Constraints added for 9 units.


In [61]:
n.model.constraints['StorageUnit-intraday_soc_min']

Constraint `StorageUnit-intraday_soc_min` [StorageUnit: 9, hour: 24]:
---------------------------------------------------------------------
[hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA, 0]: +1 StorageUnit-state_of_charge[(2040, 2040-01-01 00:00:00), hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA] + 1 StorageUnit-state_of_charge[(2040, 2040-01-02 00:00:00), hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA] + 1 StorageUnit-state_of_charge[(2040, 2040-01-03 00:00:00), hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA] ... +1 StorageUnit-state_of_charge[(2040, 2040-12-29 00:00:00), hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA] + 1 StorageUnit-state_of_charge[(2040, 2040-12-30 00:00:00), hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA] + 1 StorageUnit-state_of_charge[(2040, 2040-12-31 00:00:00), hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA]  ≥ 7372197.0
[hydro-pumped-storage-unspecified:GRIDREGION-JPN-KA, 1]: +1 StorageUnit-state_of_charge[(2040, 2040-01-01 01:0

In [ ]:
n.model.constraints['hydro_pumped_storage_unspecified_GRIDREGION_JPN_KY_avg_soc_max_h23']

In [ ]:
n.model.constraints['hydro-pumped-storage-unspecified:GRIDREGION-JPN-CB_avg_soc_min_h0']

In [ ]:
bounds

In [62]:
n.optimize.solve_model(
    solver_name='gurobi',
    # solver_options={
    #     'threads': 8,
    #     'method': 2, # barrier
    #     'crossover': 0,
    #     'BarConvTol': 1.e-6,
    #     'Seed': 123,
    #     'AggFill': 0,
    #     'PreDual': 0,
    # },
    # io_api="direct",
    # env=None,
)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00,  5.35it/s]
INFO:linopy.io: Writing time: 7.58s


Set parameter WLSAccessID


INFO:gurobipy:Set parameter WLSAccessID


Set parameter WLSSecret


INFO:gurobipy:Set parameter WLSSecret


...

INFO:gurobipy:Solved in 441172 iterations and 45.64 seconds (76.82 work units)


Optimal objective  3.906414150e+10


INFO:gurobipy:Optimal objective  3.906414150e+10


INFO:gurobipy:Warning: environment still referenced so free is deferred (Continue to use WLS)
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 1217640 primals, 2908781 duals
Objective: 3.91e+10
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-fix-p-ramp_limit_up, Generator-fix-p-ramp_limit_down, Link-fix-p-lower, Link-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-state_of_charge_set, StorageUnit-energy_balance were not assigned to the network.


('ok', 'optimal')

In [ ]:
n.storage_units[['p_max_pu', 'p_min_pu']]

In [ ]:
n.storage_units_t.p_dispatch.sum() / (n.storage_units.p_nom * 8760)

In [ ]:
n.storage_units_t.p_store.sum() / (n.storage_units.p_nom * 8760)


In [ ]:
n.storage_units_t.p_dispatch.sum() / n.storage_units_t.p_store.sum()

In [ ]:
import plotly.express as px

px.line(n.storage_units_t.state_of_charge.filter(regex='GRIDREGION-JPN-HK').reset_index(drop=True, level=0))

In [ ]:
n.links_t.p0.sum() / (n.links.p_nom * 8760)

In [ ]:
n.links_t.p0.sum()

In [ ]:
n.storage_units_t.state_of_charge

In [63]:
n.export_to_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-OCCTO-003_testing-v4/platform_network.solved.nc")

INFO:pypsa.io:Exported network 'platform_network.solved.nc' contains: storage_units, generators, loads, buses, links, carriers


<xarray.Dataset> Size: 21MB
Dimensions:                                       (snapshots: 8760,
                                                   investment_periods: 1,
                                                   storage_units_i: 9,
                                                   storage_units_t_p_i: 9,
                                                   storage_units_t_p_dispatch_i: 9,
                                                   storage_units_t_p_store_i: 9,
                                                   ...
                                                   buses_t_p_i: 9,
                                                   buses_t_marginal_price_i: 9,
                                                   links_i: 22,
                                                   links_t_p0_i: 18,
                                                   links_t_p1_i: 18,
                                                   carriers_i: 1)
Coordinates: (12/22)
  * snapshots                                     (snapshots) int64 70kB 0 .....
  * investment_periods                            (investment_periods) int64 8B ...
  * storage_units_i                               (storage_units_i) object 72B ...
  * storage_units_t_p_i                           (storage_units_t_p_i) object 72B ...
  * storage_units_t_p_dispatch_i                  (storage_units_t_p_dispatch_i) object 72B ...
  * storage_units_t_p_store_i                     (storage_units_t_p_store_i) object 72B ...
    ...                                            ...
  * buses_t_p_i                                   (buses_t_p_i) object 72B 'G...
  * buses_t_marginal_price_i                      (buses_t_marginal_price_i) object 72B ...
  * links_i                                       (links_i) object 176B 'tran...
  * links_t_p0_i                                  (links_t_p0_i) object 144B ...
  * links_t_p1_i                                  (links_t_p1_i) object 144B ...
  * carriers_i                                    (carriers_i) object 8B 'ele...
Data variables: (12/66)
    snapshots_period                              (snapshots) int64 70kB 2040...
    snapshots_timestep                            (snapshots) datetime64[ns] 70kB ...
    snapshots_objective                           (snapshots) float64 70kB 1....
    snapshots_generators                          (snapshots) float64 70kB 1....
    snapshots_stores                              (snapshots) float64 70kB 1....
    investment_periods_objective                  (investment_periods) float64 8B ...
    ...                                            ...
    links_p_nom_opt                               (links_i) float64 176B 2.8e...
    links_opex_fixed                              (links_i) float64 176B 0.0 ...
    links_max_utilisation_rate                    (links_i) float64 176B 0.83...
    links_min_utilisation_rate                    (links_i) float64 176B nan ...
    links_t_p0                                    (snapshots, links_t_p0_i) float64 1MB ...
    links_t_p1                                    (snapshots, links_t_p1_i) float64 1MB ...
Attributes:
    network__linearized_uc:      0
    network__multi_invest:       0
    network_name:                client-earth:CE-A-OCCTO-003:v3
    network_objective:           39064141495.39528
    network_objective_constant:  0
    network_pypsa_version:       0.33.0
    network_srid:                4326
    crs:                         {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"Worl...
    meta:                        {}